# FungMod advanced capabilities: provenance to solver-time thermodynamics

This notebook connects the deepest currently implemented public
surfaces in one reproducible workflow:

1. offline-first SABIO-RK source provenance and review-only proposals;
2. registry-backed uncertainty-aware Reaction 618 simulation;
3. provenance-bound competitive and substrate-inhibition laws;
4. explicit dynamic reaction quotients, Gibbs energy, electron-balance
   binding, and solver-time forward-rate enforcement;
5. conservation, entropy-rate, solver, report, and manifest artifacts.

**Scientific boundary:** source proposals require curator review, and
every configured mechanism example is either exploratory or an
artificial framework benchmark.

**Validation:** Software execution is not empirical validation.


In [ ]:
import csv
import json
import os
from pathlib import Path

import fungmod as fm

OUTPUT_ROOT = Path(
    os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", "outputs/notebooks")
).resolve()
OUTPUT = OUTPUT_ROOT / "21_advanced_capabilities"
SAMPLE_COUNT = int(os.environ.get("FUNGMOD_NOTEBOOK_SAMPLES", "12"))
OUTPUT.mkdir(parents=True, exist_ok=True)

{"fungmod_version": fm.__version__, "output_directory": str(OUTPUT)}


## 1. Start with frozen source evidence, not an invented parameter

The default SABIO-RK path is offline-first. It reads the frozen,
checksummed Reaction 618 snapshot shipped with FungMod. The proposal is
written outside `data_registry/` and remains review-only.


In [ ]:
proposal = fm.source_proposal(provider="sabiork", reaction_id="618")
proposal_output = proposal.write(OUTPUT / "source_proposal")
proposed_records = proposal.proposed_records()

{
    "query": proposal.source_query,
    "snapshot": proposal.source_snapshot_path,
    "status": proposal.proposal_status,
    "reaction_record_count": len(proposal.reaction_records),
    "parameter_record_count": len(proposed_records["parameter_records"]),
    "written_files": sorted(path.name for path in proposal_output.paths.values()),
}


## 2. Run the registry-backed Reaction 618 uncertainty screen

The selected kinetic records are provenance-backed, while enzyme
concentration remains an explicit user-supplied exploratory prior.
Therefore the run is exploratory and its quantiles are not calibrated
confidence intervals.


In [ ]:
reaction_study = fm.virtual_experiment(
    fungi="beta-glucosidase source",
    substrates="cellobiose",
    environments="SABIO-RK Reaction 618 selected assay conditions",
)
reaction_result = reaction_study.simulate(
    mode="exploratory",
    n_samples=SAMPLE_COUNT,
    seed=618,
    output_dir=OUTPUT / "reaction_618",
    quicklook=True,
)
reaction_result.write_report(
    OUTPUT / "reaction_618" / "report",
    include_html=True,
    include_index=True,
)

{
    "preflight": [row.to_dict() for row in reaction_result.preflight_reports],
    "final_metrics": reaction_result.final_metrics()[:8],
    "threshold_times": reaction_result.threshold_times()[:6],
}


In [ ]:
sampled = reaction_result.sampled_parameters()
uncertainty = reaction_result.uncertainty_summary()
provenance = reaction_result.provenance()
suggestions = reaction_result.suggested_experiments()

{
    "exploratory_enzyme_prior": [
        row for row in sampled
        if row.get("symbol") == "enzyme_concentration_beta_glucosidase"
    ][:4],
    "uncertainty_rows": uncertainty[:6],
    "provenance_rows": provenance[:6],
    "suggested_experiments": suggestions[:6],
}


## 3. Exercise two provenance-bound inhibition laws

These homogeneous configurations are artificial software benchmarks.
They demonstrate that inhibition state ownership, positive
unit-compatible constants, primary-source metadata, maturity labels,
assumptions, and limitations all survive assembly and output writing.
They do not provide organism-specific inhibition evidence.


In [ ]:
inhibition_configs = {
    "competitive": "toy_homogeneous_competitive_inhibition.yml",
    "substrate": "toy_homogeneous_substrate_inhibition.yml",
}
inhibition_results = {}
for label, filename in inhibition_configs.items():
    inhibition_results[label] = fm.run_configured_model(
        fm.example_data_path(Path("model_configs") / filename),
        output_dir=OUTPUT / f"{label}_inhibition",
    )

inhibition_summary = {
    label: {
        "processes": sorted(run.process_rates),
        "validation": run.validation_report(),
        "solver_success": run.solver_metadata["success"],
    }
    for label, run in inhibition_results.items()
}
inhibition_summary


## 4. Run explicit solver-time thermodynamic enforcement

The packaged configuration is a generic A-to-B framework benchmark. It
supplies every activity, concentration, temperature, standard-energy,
gas-constant, electron-balance, tolerance, and provenance input
explicitly. The solver blocks an unfavorable nonnegative forward rate;
it does not infer missing chemistry.


In [ ]:
thermo_output = OUTPUT / "dynamic_thermodynamics"
thermo_result = fm.run_configured_model(
    fm.example_data_path(
        "model_configs/showcase_dynamic_thermodynamics.yml"
    ),
    output_dir=thermo_output,
)

prefix = "dynamic_thermodynamics.a_to_b_dynamic"
reaction_quotient = thermo_result.derived_quantities[
    f"{prefix}.reaction_quotient"
].magnitude
delta_gibbs = thermo_result.derived_quantities[
    f"{prefix}.delta_gibbs"
].to("joule / mole").magnitude
rate_blocked = thermo_result.derived_quantities[
    f"{prefix}.rate_blocked"
].magnitude

{
    "initial_reaction_quotient": float(reaction_quotient[0]),
    "final_reaction_quotient": float(reaction_quotient[-1]),
    "initial_delta_gibbs_J_per_mol": float(delta_gibbs[0]),
    "final_delta_gibbs_J_per_mol": float(delta_gibbs[-1]),
    "blocked_time_points": int(rate_blocked.sum()),
    "solver_dynamic_thermodynamics": thermo_result.solver_metadata[
        "dynamic_thermodynamics"
    ],
}


## 5. Inspect package-generated advanced diagnostics

The entropy-rate output uses a separately supplied static,
condition-specific delta G and explicit control-volume conversion. It
must not be confused with the dynamic delta-G trajectory used by the
solver constraint.


In [ ]:
thermodynamic_summary = json.loads(
    (thermo_output / "thermodynamic_summary.json").read_text(encoding="utf-8")
)
conservation_summary = json.loads(
    (thermo_output / "conservation_diagnostics.json").read_text(encoding="utf-8")
)
solver_summary = json.loads(
    (thermo_output / "solver_diagnostics.json").read_text(encoding="utf-8")
)
entropy_summary = json.loads(
    (thermo_output / "entropy_production_rate_timeseries.json").read_text(
        encoding="utf-8"
    )
)
with (thermo_output / "entropy_production_rate_timeseries.csv").open(
    newline="", encoding="utf-8"
) as handle:
    entropy_rows = list(csv.DictReader(handle))

{
    "thermodynamic_summary": thermodynamic_summary,
    "conservation_status_counts": conservation_summary["status_counts"],
    "solver_status": solver_summary["status"],
    "entropy_guardrail": entropy_rows[0]["guardrails"],
    "entropy_row_count": entropy_summary["row_count"],
}


## 6. Verify the advanced output bundle

The manifest closes the loop from configuration to artifacts. A
downstream analysis can discover the complete bundle without scraping
notebook output.


In [ ]:
manifest = json.loads(
    (thermo_output / "output_manifest.json").read_text(encoding="utf-8")
)
required = {
    "configured_metadata.json",
    "conservation_diagnostics.csv",
    "derived_quantities.csv",
    "entropy_production_rate_timeseries.csv",
    "process_rates.csv",
    "solver_diagnostics.csv",
    "thermodynamic_summary.csv",
    "validation_report.json",
}
missing = sorted(required - set(manifest["files"]))
assert not missing, missing
assert thermodynamic_summary["has_solver_time_enforcement"] is True
assert entropy_summary["has_dynamic_delta_gibbs"] is False

{
    "artifact_count": len(manifest["files"]),
    "missing_required_artifacts": missing,
    "validation": thermo_result.validation_report(),
}


## Capability boundary

This notebook exercised deep implemented contracts, but it intentionally
did not:

- promote a source proposal without an explicit curator decision;
- present artificial inhibition or thermodynamic inputs as biology;
- claim empirical validation, calibration, or prediction accuracy;
- infer nonideal activities, reverse rates, coupled-network fluxes,
  intracellular metabolism, or whole-fungus physiology.

Those omissions are scientific controls, not missing notebook polish.
